# Tuning! 

- ***note***: 
  - Unfortunately, the tuning steps are not replicable since using n_jobs > 1 on the optuna optimizer will never output replicable results (even if a sampler with a fixed seed is provided)
  - Do not run the cells 

In [1]:
import pandas as pd
import numpy as np 
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV
from optuna.samplers import RandomSampler
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier
from sklearn.base import clone
from feature_eng import pipe_non_tree_new_eng, pipe_boosted_tree_new_neg, pipe_random_tree_new_eng 
from lightgbm import LGBMClassifier
import warnings
from xgboost import XGBClassifier

c:\Users\39320\AppData\Local\Programs\Python\Python312\Lib\site-packages\cupy\_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


In [ ]:
dev = pd.read_csv("development(1).csv")
ev = pd.read_csv("evaluation.csv")

y = dev["death"]
x = dev.drop(columns=["death"])
cat_indices = [i for i, _ in enumerate(x.drop(columns=["temp_f", "Id", "adls"])) if _ in {"sex", "medical_center", "dzgroup", "dzclass", 
                                                           "race", "ca", "dnr"}]

In [ ]:

# fix for a catboost bug! 
class CustomCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return CustomCatBoostClassifier(**self.get_params())

In [ ]:
# [I 2026-09-09 00:26:55,328] Trial 58 finished with value: 0.7507453798891423 and parameters: {'C': 0.052276215026302975, 
# 'l1_ratio': 0.851844421406245, max_iter: 5000}. Best is trial 58 with value: 0.7507453798891423.
#     CV                   THRESH
# 0.7507453798891423 0.5420956679213134


def objective(trial):
    param = {
        "C": trial.suggest_float("C",  0.001, 1000.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0, 1),
        "max_iter": 5000
    }
# make_pipeline(pipe_boosted_tree, model)
    model = LogisticRegression(random_state=17, solver="saga",  **param)
    m = TunedThresholdClassifierCV(make_pipeline(pipe_non_tree_new_eng, model), scoring="f1_macro", 
                             cv=StratifiedKFold(n_splits=7, shuffle=True, random_state=17), n_jobs=7)
    m.fit(x, y)
    print(m.best_score_, m.best_threshold_)
    return m.best_score_


study = optuna.create_study(direction='maximize', sampler=RandomSampler(17))
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=100, n_jobs=2)

In [ ]:
# [I 2026-09-09 13:00:42,436] Trial 78 finished with value: 0.7577273510746684 and parameters: 
# {'n_estimators': 4717, 'learning_rate': 0.005221174518290643, 'min_child_weight': 1, 'max_depth': 2, 
# 'subsample': 0.9099870866208125, 'colsample_bytree': 0.8838338623066402, 
# 'colsample_bylevel': 0.8534163510022217, 'max_bin': 195, 'reg_lambda': 1.9188323258235047, 
# 'reg_alpha': 0.00025552830819887515, 'gamma': 0.00019769506514463376, 'scale_pos_weight': 0.39188495395584216}. Best is trial 78 with value: 0.7577273510746684.
#     CV               THRESH
# 0.7577273510746684 0.36024192

# 1. Define an objective function to be maximized.
def objective(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
        "learning_rate": trial.suggest_float("learning_rate",  0.001, 1.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "max_depth": trial.suggest_int("max_depth",1, 12),
        "subsample": trial.suggest_float("subsample",0.1, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.1, 1.0),
        "max_bin": trial.suggest_int("max_bin", 32, 512),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.3, 0.6, log=True),
    }
    m = TunedThresholdClassifierCV(make_pipeline(pipe_boosted_tree_new_neg,  
                                                      XGBClassifier(**param, device="cuda", random_state=17, 
                                                                             tree_method="hist", enable_categorical=True, n_jobs=4)), scoring="f1_macro", 
                                 cv=StratifiedKFold(n_splits=7, shuffle=True, random_state=17), n_jobs=7)

    
    m.fit(x, y)
    print(m.best_score_, m.best_threshold_)
    return m.best_score_


  
study = optuna.create_study(direction='maximize', sampler=RandomSampler(17))
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=200, n_jobs=4)

In [ ]:
# [I 2026-09-09 06:48:33,036] Trial 108 finished with value: 0.7609402917581694 and parameters: 
# {'iterations': 4174, 'learning_rate': 0.0040932218353216246, 'random_strength': 0.001562384211665802, 'depth': 6, 
# 'bagging_temperature': 0.8156555071877585, 'border_count': 214, 'l2_leaf_reg': 19, 'scale_pos_weight': 0.47919356417193015}. Best is trial 108 with value: 0.7609402917581694.
#    CV SCORE            THRESH
# 0.7609402917581694 0.45082660405796093
# logging_level="Silent",cat_features=cat_indices, random_state=17

def objective(trial):
    param = {
        "iterations": trial.suggest_int("iterations", 10, 5000),
        "learning_rate": trial.suggest_float("learning_rate",  0.001, 1.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10.0, log=True),
        "depth": trial.suggest_int("depth", 1, 8),
        "bagging_temperature": trial.suggest_float("bagging_temperature",0.0, 1.0),
        "border_count":  trial.suggest_int("border_count", 1, 255),
        "l2_leaf_reg":  trial.suggest_int("l2_leaf_reg", 2, 30),
        "scale_pos_weight":  trial.suggest_float("scale_pos_weight", 0.3, 0.6),
    }

    m = TunedThresholdClassifierCV(make_pipeline(pipe_boosted_tree_new_neg, CustomCatBoostClassifier(**param,  logging_level="Silent",
                           cat_features=cat_indices, random_state=17)), cv=StratifiedKFold(7, random_state=17, shuffle=True), n_jobs=7, scoring="f1_macro")

    m.fit(x, y)
    print(m.best_score_, m.best_threshold_)

    
    return m.best_score_

study = optuna.create_study(direction='maximize', sampler=RandomSampler(17))
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=300, n_jobs=4)

In [ ]:

# [I 2026-09-10 10:55:26,389] Trial 84 finished with value: 0.7539440441665058 and parameters: 
# {'max_features': 'log2', 'max_depth': 28, 'min_samples_leaf': 3, 'n_estimators': 4651, 'class_weight': 'balanced'}. Best is trial 84 with value: 0.7539440441665058.
#      CV                THRESH
# 0.7539440441665058 0.545837648927173


def objective(trial):
    param = {
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.1, 1/20]),
        "max_depth": trial.suggest_int("max_depth",1, 32),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf",1, 30),
        "n_estimators":  trial.suggest_int("n_estimators", 100, 5000),
        "class_weight":  trial.suggest_categorical("class_weight", ["balanced"]),
    }
    m = TunedThresholdClassifierCV(make_pipeline(pipe_random_tree_new_eng, RandomForestClassifier(**param,
                           random_state=17, n_jobs=5)), scoring="f1_macro", 
                                 cv=StratifiedKFold(n_splits=7, shuffle=True, random_state=17), n_jobs=7)
    m.fit(x, y)
    print(m.best_score_, m.best_threshold_)
    return m.best_score_


study = optuna.create_study(direction='maximize', sampler=RandomSampler(17))
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=100, n_jobs=2)

In [ ]:
#  {'n_estimators': 3342, 'learning_rate': 0.0028954614575091887, 'min_data_in_leaf': 81, 
#                           'max_depth': 9, 'min_gain_to_split': 1.3209989283293588, 
#                           'subsample': 0.8822893771001694, 'subsample_freq': 2, 
#                           'max_bin': 238, 'feature_fraction': 0.6511238486969437, 
#                           'reg_lambda': 0.04227276364865463, 
#                           'reg_alpha': 0.8086097224250781, 
#                           'scale_pos_weight': 0.49326982842529654, 
#                           'num_leaves': 198}
warnings.filterwarnings("ignore", category=UserWarning)
def objective(trial):
    param = {
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 10000),
        "learning_rate": trial.suggest_float("learning_rate",  0.001, 1.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 0, 300),
        "max_depth": trial.suggest_int("max_depth",1, 12),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 15.0),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 10),
        "max_bin": trial.suggest_int("max_bin", 32, 512),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.001, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.001, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.3, 0.6, log=True),
    }
    param["num_leaves"] = trial.suggest_int("num_leaves", 2, 2 ** param["max_depth"])
   
    ths = TunedThresholdClassifierCV(make_pipeline(pipe_boosted_tree_new_neg , LGBMClassifier(**param,
                           random_state=17)), scoring="f1_macro", cv=StratifiedKFold(7, shuffle=True, random_state=17), n_jobs=7)
    ths.fit(x, y)
    print(ths.best_threshold_, ths.best_score_)
    return ths.best_score_ 


study = optuna.create_study(direction='maximize', sampler=RandomSampler(17))
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=200, n_jobs=4)